In [10]:
from pathlib import Path
import json, csv
from datetime import date, timedelta
from typing import Dict, List, Iterable, Tuple, Optional
from collections import defaultdict
PROJECT = Path(__file__).resolve().parents[2] if '__file__' in globals() else Path.cwd().parents[2]
RAW = PROJECT / "data" / "raw" / "tomtom_stats"
INPUT_JSON = RAW / "8285492_HCMC_CBD_Weekday_07-10___15-18__FRC0-7__unzipped.json"
GRAPH_DIR = PROJECT / "data" / "raw" / "tomtom_stats" / "graphs"
PROC = RAW / "times"
PROC.mkdir(parents=True, exist_ok=True)

OUT_OBS   = PROC / "observations.csv"
OUT_TS    = PROC / "time_sets.csv"
OUT_DAYS  = PROC / "days_by_dateRange.csv"
INPUT     = INPUT_JSON  # giữ nguyên biến của bạn
TARGET_FRC: Optional[set] = None


In [11]:
# ----- đọc JSON -----
def _split_tr(tr: str) -> Tuple[str, str]:
    a, b = tr.split("-")
    return a, b

def _percentiles(sp: List[int]) -> Tuple:
    return (
        sp[2]  if len(sp) >= 3  else "",
        sp[5]  if len(sp) >= 6  else "",
        sp[9]  if len(sp) >= 10 else "",
        sp[13] if len(sp) >= 14 else "",
        sp[17] if len(sp) >= 18 else "",
    )

# ---------- load JSON ----------
with open(INPUT, "r", encoding="utf-8") as f:
    data = json.load(f)

# GIỮ NGUYÊN ID DẠNG STRING (KHÔNG CONVERT)
date_ranges: Dict[str,dict] = {dr["@id"]: dr for dr in data["dateRanges"]}
time_sets  : Dict[str,dict] = {ts["@id"]: ts for ts in data["timeSets"]}
segments   : List[dict]     = data["network"]["segmentResults"]


In [12]:
# ----- timeSet → {valid_dow, time_range} -----
timeset_to_valid_dow: Dict[str, set] = {}
timeset_to_timerange: Dict[str, str] = {}

for tid, ts in time_sets.items():
    dows = {d["dayOfWeek"].upper() for d in ts["dayToTimeRanges"]}
    tr   = ts["dayToTimeRanges"][0]["timeRanges"][0]  # always 1 slot in TomTom
    timeset_to_valid_dow[tid] = dows
    timeset_to_timerange[tid] = tr


In [13]:
# ---- DateRange → list of (date, dow) ----
def _list_days(dr: dict) -> List[Tuple[str,str]]:
    y1,m1,d1 = map(int, dr["from"].split("-"))
    y2,m2,d2 = map(int, dr["to"].split("-"))

    cur, end = date(y1,m1,d1), date(y2,m2,d2)
    ex_week = set(dr.get("excludedDaysOfWeek", []))

    out = []
    while cur <= end:
        dow = cur.strftime("%A").upper()
        if dow not in ex_week:
            out.append((cur.isoformat(), dow))
        cur += timedelta(days=1)

    return out

days_by_dr = {drid: _list_days(dr) for drid, dr in date_ranges.items()}


In [14]:
cnt_seen_frc      = defaultdict(int)
cnt_written_frc   = defaultdict(int)
drop_no_frc       = 0
drop_not_in_set   = 0
drop_no_timeRes   = 0
drop_no_timeset   = 0
drop_no_days      = 0
drop_dow_mismatch = 0

with open(OUT_OBS, "w", newline="", encoding="utf-8") as fo:
    w = csv.writer(fo)
    w.writerow([
        "date","day_of_week","time_set_id","time_start","time_end",
        "segment_id","streetName","frc","speedLimit","distance",
        "sampleSize","averageSpeed","medianSpeed","harmonicAverageSpeed",
        "averageTravelTime","medianTravelTime","travelTimeRatio",
        "p05","p25","p50","p75","p95","is_observed"
    ])

    rows = 0
    seg_kept = 0

    for seg in segments:
        frc_raw = seg.get("frc")
        if frc_raw is None:
            drop_no_frc += 1
            continue

        try:
            frc = int(frc_raw)
        except:
            drop_no_frc += 1
            continue

        cnt_seen_frc[frc] += 1

        if TARGET_FRC is not None and frc not in TARGET_FRC:
            drop_not_in_set += 1
            continue

        sid    = seg.get("segmentId")
        street = seg.get("streetName","")
        sl     = seg.get("speedLimit","")
        dist   = seg.get("distance","")

        trs = seg.get("segmentTimeResults") or []
        if not trs:
            drop_no_timeRes += 1
            continue

        wrote_any_for_seg = False

        for tr in trs:
            tid  = tr.get("timeSet")      # KEEP STRING ID
            drid = tr.get("dateRange")    # KEEP STRING ID

            trange = timeset_to_timerange.get(tid)
            if trange is None:
                drop_no_timeset += 1
                continue

            ts, te = _split_tr(trange)
            valid_dow = timeset_to_valid_dow.get(tid, set())

            p05,p25,p50,p75,p95 = _percentiles(tr.get("speedPercentiles") or [])
            sample = tr.get("sampleSize", 0) or 0

            days = days_by_dr.get(drid, [])
            if not days:
                # fallback với missing dateRange
                w.writerow([
                    "", "", tid, ts, te,
                    sid, street, frc, sl, dist,
                    sample,
                    tr.get("averageSpeed",""),
                    tr.get("medianSpeed",""),
                    tr.get("harmonicAverageSpeed",""),
                    tr.get("averageTravelTime",""),
                    tr.get("medianTravelTime",""),
                    tr.get("travelTimeRatio",""),
                    p05,p25,p50,p75,p95,
                    1 if sample > 0 else 0
                ])
                rows += 1
                wrote_any_for_seg = True
                drop_no_days += 1
                continue

            wrote_this_tr = False
            for d, dow in days:
                if dow in valid_dow:
                    w.writerow([
                        d, dow, tid, ts, te,
                        sid, street, frc, sl, dist,
                        sample,
                        tr.get("averageSpeed",""),
                        tr.get("medianSpeed",""),
                        tr.get("harmonicAverageSpeed",""),
                        tr.get("averageTravelTime",""),
                        tr.get("medianTravelTime",""),
                        tr.get("travelTimeRatio",""),
                        p05,p25,p50,p75,p95,
                        1 if sample > 0 else 0
                    ])
                    rows += 1
                    wrote_any_for_seg = True
                    wrote_this_tr = True

            if not wrote_this_tr and days:
                drop_dow_mismatch += 1

        if wrote_any_for_seg:
            seg_kept += 1
            cnt_written_frc[frc] += 1


In [15]:
print(f"✅ Saved: {OUT_OBS}")
print(f"Segments seen by FRC:    {dict(sorted(cnt_seen_frc.items()))}")
print(f"Segments written by FRC: {dict(sorted(cnt_written_frc.items()))}")
print(f"Segments kept: {seg_kept} | Rows: {rows}")

print("\n--- Diagnostics (skip reasons) ---")
print(f"no FRC:            {drop_no_frc}")
print(f"filtered by set:   {drop_not_in_set}")
print(f"no timeResults:    {drop_no_timeRes}")
print(f"no timeset map:    {drop_no_timeset}")
print(f"no days for drid:  {drop_no_days}")
print(f"DOW mismatch:      {drop_dow_mismatch}")


✅ Saved: C:\AI\Specialized_Project2_github\Urban-Traffic-Links\data\raw\tomtom_stats\times\observations.csv
Segments seen by FRC:    {2: 1002, 3: 672, 4: 941, 6: 2802, 7: 5220}
Segments written by FRC: {2: 1002, 3: 672, 4: 941, 6: 2802, 7: 5220}
Segments kept: 10637 | Rows: 5616336

--- Diagnostics (skip reasons) ---
no FRC:            0
filtered by set:   0
no timeResults:    0
no timeset map:    0
no days for drid:  0
DOW mismatch:      0


In [16]:
import pandas as pd
from pathlib import Path

OBS = PROC / "observations.csv"
SEG = GRAPH_DIR / "segments.csv"

obs = pd.read_csv(OBS, usecols=["segment_id","streetName"]).drop_duplicates("segment_id")
seg = pd.read_csv(SEG, usecols=["segment_id","streetName","frc","speedLimit","distance"]).drop_duplicates("segment_id")

only_in_seg = seg.merge(obs[["segment_id"]], on="segment_id", how="left", indicator=True)
only_in_seg = only_in_seg[only_in_seg["_merge"]=="left_only"].drop(columns="_merge")

only_in_obs = obs.merge(seg[["segment_id"]], on="segment_id", how="left", indicator=True)
only_in_obs = only_in_obs[only_in_obs["_merge"]=="left_only"].drop(columns="_merge")

print("🚩 Có trong segments.csv nhưng KHÔNG có trong observations.csv:", len(only_in_seg))
print(only_in_seg.head(10))
print("🚩 Có trong observations.csv nhưng KHÔNG có trong segments.csv:", len(only_in_obs))
print(only_in_obs.head(10))


🚩 Có trong segments.csv nhưng KHÔNG có trong observations.csv: 0
Empty DataFrame
Columns: [segment_id, streetName, frc, speedLimit, distance]
Index: []
🚩 Có trong observations.csv nhưng KHÔNG có trong segments.csv: 182
          segment_id                 streetName
60   -17040020964331         Đường Phan Văn Trị
230  -17040020002244           Đường Hoàng Diệu
305  -17040019307666           Đường Vĩnh Khánh
316  -17040024165470  Hẻm TK28 Nguyễn Cảnh Chân
679  -17040020104054           Đường Cống Quỳnh
752  -17040021911648       Chung Cư Đoàn Văn Bơ
769  -17040022666835          Đường Nguyễn Trãi
1043 -17040018939576  Hẻm 214 Nguyễn Văn Nguyễn
1244 -17040023842841              Đường Đề Thám
1282 -17040019337594                 Đường Số 9
